# Phase 1: Tokenization & Text Preprocessing

Welcome to the first notebook of your NLP learning journey! This repository is designed to be an educational resource. Rather than just running code, we walk through the algorithmic choices and the practical implementation details of each concept.

## What is Preprocessing, and Why Do We Need It?
Raw text is a sequence of characters. Machine learning algorithms, however, operate on numbers. The bridge between raw text and mathematical vectors is **text preprocessing**.

In this phase, we explore the following key stages of the preprocessing pipeline:
- **Tokenization**: Breaking continuous strings into discrete units (words, subwords, or sentences).
- **Stemming & Lemmatization**: Reducing words to their base or root forms to collapse vocabulary size.
- **Stopword Filtering**: Removing high-frequency, low-semantic-value words.
- **Regex Cleaning**: Stripping URLs, HTML entities, and formatting noise.

Let's get started!

---  
# Part 1: From-Scratch Implementations

By building these algorithms yourself, you will gain a deep understanding of the edge cases and complexities that library maintainers have solved for us. We will use nothing but Python's standard library.

## 1.1 Whitespace Tokenization

The most intuitive way to separate text into words is to split it whenever we encounter a space, tab, or newline character.

### Challenges
1. **Punctuation Attachment**: Punctuation symbols (like `!`, `,`, `.`) remain attached to the word immediately preceding them. In NLP, `perfectly.` and `perfectly` should generally be treated as the same word representation, not two separate vocabulary items.
2. **Contractions**: Words like `doesn't` or `it's` represent multiple semantic elements (`does` + `not`, `it` + `is`). Whitespace splitting keeps them bound together.
3. **Hyphenation**: Compounds like `state-of-the-art` might need to be split into separate tokens or kept unified depending on the downstream application.

### Task
Implement a basic function that splits a string on any whitespace characters.

In [ ]:
def tokenize_whitespace(text):
    """
    Splits text by whitespace characters (spaces, tabs, newlines).
    
    Args:
        text (str): The input text to tokenize.
        
    Returns:
        list: A list of tokens.
    """
    # TODO: Implement whitespace splitting (Hint: look at Python's str.split())
    pass

In [ ]:
# Test case
sample_text = "NLP is state-of-the-art! However, it doesn't always work perfectly."
tokens = tokenize_whitespace(sample_text)
print("Whitespace Tokens:", tokens)

# Note: Observe how 'state-of-the-art!', 'However,', and 'perfectly.' retain punctuation.
# Contractions like "doesn't" are kept as a single token.

## 1.2 Regex-Based Word Tokenization

To solve the punctuation issue, we can use regular expressions. By defining patterns, we can target alphanumeric character sequences separately from special punctuation characters.

### How the Pattern Works
We can search for words using the pattern `\w+(?:'\w+)?`. This captures:
- `\w+`: One or more word characters (letters, numbers, underscores).
- `?:'\w+)?`: An optional non-capturing group for apostrophes followed by word characters (e.g., matching the `'t` in `doesn't`).
- Alternatively, we match any single punctuation character using standard punctuation sets.

### Task
Implement `tokenize_regex` using Python's `re.findall` to extract words and punctuation separately.

In [ ]:
import re

def tokenize_regex(text):
    """
    Tokenizes text using regular expressions.
    Should separate punctuation from words, while keeping contractions intact.
    
    Args:
        text (str): The input text to tokenize.
        
    Returns:
        list: A list of tokens with punctuation isolated.
    """
    # TODO: Define a pattern and extract tokens using re.findall
    # A standard starting pattern: word characters with optional apostrophe, OR any punctuation mark.
    pattern = r"\w+(?:'\w+)?|[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~]"
    return re.findall(pattern, text)

In [ ]:
print("Regex Tokens:", tokenize_regex(sample_text))
# Verify that punctuation marks like '!' and '.' are now distinct tokens.

## 1.3 Simple Sentence Tokenization

A document contains paragraphs, paragraphs contain sentences, and sentences contain words. For tasks like summarization, translation, or semantic parsing, we need to process text at the sentence level.

### The Challenge of Ambiguity
We cannot simply split text at every occurrence of a period `.`, exclamation mark `!`, or question mark `?`. Consider:
- Abbreviations: `Dr. Smith`, `Mr. Jones`, `e.g.`, `Inc.`
- Decimal values: `$1,000.50` or `3.14`
- Quotation marks: `"Hello!" she said.` (the exclamation is inside a quote but might not end the main sentence context)

### Task
Write a basic rule-based sentence tokenizer that splits on sentence boundary punctuation while attempting to ignore common abbreviations.

In [ ]:
def split_sentences(text):
    """
    Splits text into sentences, handling common sentence-ending punctuation (.?!)
    while attempting to avoid splitting on common abbreviations like Dr., Mr., etc.
    
    Args:
        text (str): The input string to split into sentences.
        
    Returns:
        list: A list of sentence strings.
    """
    # TODO: Implement basic sentence boundary detection.
    # One simple method is using regex split: e.g., re.split(r'(?<!\bDr)(?<!\bMr)[.!?]\s+', text)
    # Review where it works and where it fails.
    pass

In [ ]:
sentence_sample = "Dr. Smith bought a laptop. It cost $1,000.50. What a deal!"
print("Sentences:", split_sentences(sentence_sample))

## 1.4 Basic Suffix Stemming

A single base concept can have multiple surface representations (e.g., `connect`, `connects`, `connecting`, `connection`, `connected`). Stemming attempts to collapse these variants down to a single root representation (or "stem") by removing suffixes. This reduces the overall size of the vocabulary that our downstream models need to learn.

### How Stemming Works
Stemmers use a heuristic set of rules. For example, in the Porter Stemmer:
- If a word ends in `sses`, replace with `ss` (e.g., `caresses` -> `caress`)
- If a word ends in `ies`, replace with `i` (e.g., `ponies` -> `poni`)
- Strip suffix endings like `ing`, `ed`, `ly`, or `s`.

### Task
Create a simple heuristic stemmer function that checks for common suffixes and strips them.

In [ ]:
def simple_stemmer(word):
    """
    Reduces a word to its root by stripping common suffixes like -ing, -ed, -ly, -s.
    
    Args:
        word (str): The input word.
        
    Returns:
        str: The stemmed word root.
    """
    # TODO: Implement basic suffix stripping rules using basic string checks (.endswith())
    pass

In [ ]:
test_words = ["connections", "connecting", "connected", "happily", "dogs"]
for w in test_words:
    print(f"{w} -> {simple_stemmer(w)}")

## 1.5 Byte Pair Encoding (BPE) from Scratch

Traditional word-level tokenizers cannot handle **Out-Of-Vocabulary (OOV)** words — words that weren't seen during training. Subword tokenization (such as Byte Pair Encoding, BPE) addresses this by breaking unfamiliar words down into smaller sub-units (e.g., `un-`, `predict-`, `-able`). BPE is the standard tokenization methodology for state-of-the-art LLMs (GPT-2/3/4, Llama).

### BPE Algorithmic Flow
1. **Pre-tokenize the corpus**: Split sentences into words and represent them as lists of individual characters, appending a special end-of-word marker `</w>`.
2. **Count pairs**: Analyze the corpus to find the most frequent adjacent pair of symbols (e.g., `('e', 'r')`).
3. **Merge**: Combine this pair into a new symbol (e.g., `'er'`) everywhere in the vocabulary.
4. **Repeat**: Repeat this merging process for a fixed number of iterations. Each iteration expands our vocabulary by one merge rule.

### Task
Review the helper functions below and complete the loop to perform BPE merging over a sample corpus.

In [ ]:
from collections import defaultdict

def get_vocab_counts(corpus):
    """
    Converts a corpus of sentences into a dictionary of word frequencies, 
    representing words as tuples of characters with an end-of-word marker '</w>'.
    """
    vocab = defaultdict(int)
    for sentence in corpus:
        words = sentence.split()
        for word in words:
            char_tuple = tuple(list(word)) + ('</w>',)
            vocab[char_tuple] += 1
    return vocab

def get_stats(vocab):
    """
    Computes frequencies of all adjacent pairs of symbols in the vocabulary.
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab_in):
    """
    Merges all occurrences of the specified pair in the vocabulary.
    """
    vocab_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab_in:
        word_str = ' '.join(word)
        word_merged = p.sub(''.join(pair), word_str)
        vocab_out[tuple(word_merged.split())] = vocab_in[word]
    return vocab_out

In [ ]:
# Corpus definition
corpus = [
    "low low low low low lower lower newer newer newer newer newer newer"
]
vocab = get_vocab_counts(corpus)
num_merges = 10

print("Initial Vocabulary (Characters):", vocab)

# TODO: Write a loop that executes BPE merging for 'num_merges' iterations.
# 1. Calculate stats of adjacent pairs using get_stats(vocab).
# 2. Find the most frequent pair using max(pairs, key=pairs.get).
# 3. Merge the vocab using merge_vocab(best_pair, vocab).
# 4. Print the merge operation and updated vocab.
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge #{i+1}: {best} -> Vocab state: {vocab}")

---  
# Part 2: Library-Based Preprocessing

Now that you've implemented the concepts from scratch, let's learn how to leverage robust industry tools: **NLTK**, **spaCy**, and the **Hugging Face Tokenizers** ecosystem.

## 2.1 Environment Verification & Setup

Before executing the cells below, make sure you download the required resources. 
- NLTK requires downloading data bundles (like tokenizers and list of stopwords).
- spaCy requires downloading language-specific statistical models (like English `en_core_web_sm`).

In [ ]:
import nltk
import spacy

# Download NLTK data (tokenizers and stopwords)
nltk.download('punkt')
nltk.download('stopwords')

# Download spaCy English model if not already downloaded:
# In your terminal inside the virtual environment run: 
# python -m spacy download en_core_web_sm

## 2.2 Word & Sentence Tokenization with NLTK

### NLTK (Natural Language Toolkit)
NLTK is one of the oldest and most widely used educational libraries in Python for text processing. It uses heuristic algorithms (like the *Punkt* sentence tokenizer) to split words and sentences.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

sample_text = "NLP is state-of-the-art! However, it doesn't always work perfectly."

nltk_words = word_tokenize(sample_text)
nltk_sentences = sent_tokenize(sample_text)

print("NLTK Word Tokenization:", nltk_words)
print("NLTK Sentence Tokenization:", nltk_sentences)

## 2.3 Tokenization, Lemmatization, and Stopwords with spaCy

### spaCy
spaCy is built specifically for production usage. It is extremely fast and processes text by passing it through an active pipeline. When you run `nlp(text)`, spaCy tokenizes the text, parses Part of Speech (POS) tags, detects syntactic dependencies, extracts Named Entities (NER), and computes base lemmas.

In [ ]:
# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")

doc = nlp(sample_text)

# Extract tokens, lemmas, stopword status, and part-of-speech
spacy_data = []
for token in doc:
    spacy_data.append({
        "Text": token.text,
        "Lemma": token.lemma_,
        "Is Stopword": token.is_stop,
        "POS Tag": token.pos_
    })

import pandas as pd
df_spacy = pd.DataFrame(spacy_data)
df_spacy

## 2.4 Comparing Stemmers vs. Lemmatizers

### Stemming vs. Lemmatization: What is the Difference?
- **Stemming**: Uses quick, heuristic rules to cut off suffixes (e.g., `was` -> `wa`, `better` -> `bett`). It can lead to non-dictionary words.
- **Lemmatization**: Uses complete morphological analysis and a dictionary lookup (WordNet / spaCy vocabulary) to return the actual base word (e.g., `was` -> `be`, `better` -> `good`). It is slower but semantically accurate.

Let's compare them side-by-side using NLTK's Porter Stemmer and spaCy's Lemmatizer.

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

words_to_compare = ["connecting", "connection", "connections", "connected", "studies", "studying", "was", "am", "better"]

comparison = []
for word in words_to_compare:
    stem = stemmer.stem(word)
    doc_word = nlp(word)
    lemma = doc_word[0].lemma_
    
    comparison.append({
        "Word": word,
        "Porter Stem": stem,
        "spaCy Lemma": lemma
    })

df_compare = pd.DataFrame(comparison)
df_compare

## 2.5 Subword Tokenization with Hugging Face

### Hugging Face `tokenizers`
Modern language models (transformers) do not split strictly on spaces or punctuation. Instead, they use subword algorithms. Let's see how the pre-trained GPT-2 BPE tokenizer breaks up our sample text.

In [ ]:
from transformers import GPT2Tokenizer

# Load the pre-trained GPT-2 BPE tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokens = tokenizer.tokenize(sample_text)
ids = tokenizer.convert_tokens_to_ids(tokens)

for t, i in zip(tokens, ids):
    print(f"Token: {t:<15} ID: {i}")

---  
# Part 3: Hands-On Exercises

Apply your knowledge to complete these three exercises. Check your output assertions where provided!

## Exercise 3.1: Stemming vs. Lemmatization Comparison

### Objective
Process a small batch of sentences and extract the stemming/lemmatization differences. Focus on identifying:
- **Over-stemming**: Where the stemmer cuts off too much (e.g., resolving unrelated words to the same root, losing meaning).
- **Under-stemming**: Where the stemmer fails to group related words to the same root.

In [ ]:
# Define sentences with complex word forms
sentences = [
    "The children are playing happily in the garden, having fun with their toys.",
    "The studies showed that studying regularly produces better study results.",
    "He was running, jumps over the fence, and then walked slowly home.",
    "The items were bought at a discount, but some were broken or damaged."
]

# TODO: Tokenize each sentence using NLTK, compute the stem (NLTK Porter) and lemma (spaCy) for each token.
# Compile the results into a single pandas DataFrame for comparison and display it.

## Exercise 3.2: Noisy Social Media Cleaning Pipeline

### Objective
Build a robust cleaning function using regular expressions to clean messy tweets before tokenization.

In [ ]:
tweets = [
    "Check out this amazing article! https://t.co/xyz123 @user123 #NLP #AI is the future!!!",
    "RT @another_user: Python is &lt;great&gt; for data science. Go to http://python.org",
    "I    love   natural    language    processing... it's so cool!!!"
]

def clean_tweet(text):
    """
    Cleans a tweet by:
    1. Removing URLs (http:// or https:// and any following non-whitespace characters).
    2. Removing user mentions (@username).
    3. Stripping/converting HTML entities (like converting '&lt;' to '<', '&gt;' to '>', and '&amp;' to '&').
    4. Normalizing whitespace (replacing tabs, newlines, and multiple spaces with a single space, stripping boundary spaces).
    """
    # TODO: Implement the cleaning logic
    pass

for tweet in tweets:
    print("Original:", tweet)
    print("Cleaned: ", clean_tweet(tweet))
    print("-" * 50)

## Exercise 3.3: Complete Preprocessing Pipeline

### Objective
Combine cleaning, tokenization, stopword removal, and lemmatization into a single production-like function.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Takes raw text, cleans it (removes URLs/mentions, normalizes whitespace),
    tokenizes it, removes stopwords and punctuation, and returns a list of lemmatized tokens in lowercase.
    
    Args:
        text (str): The input raw string.
        
    Returns:
        list: Cleaned, tokenized, and lemmatized token strings.
    """
    # TODO: Implement complete preprocessing pipeline using spaCy and clean_tweet
    pass

In [ ]:
raw_doc = """The Quick Brown Fox jumps over the lazy dog! 
Visit http://example.com/fox for more details. @fox_fan club."""

print("Preprocessed:", preprocess_text(raw_doc))
# Expected output should be a list of lowercased lemmas, e.g.:
# ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog', 'visit', 'detail', 'club']